# CRC Xenium: prepare training subset (~30k cells)

Separate from `spatial_dataset_download_test.ipynb` (which only tested whether these datasets were reachable/what's inside). This notebook's one job: produce the final small `.h5ad` we'll actually train scProto on.

Source: Marteau et al. 2026 (Cancer Cell), CRC neutrophil spatial atlas, BioImage Archive S-BIAD2208, file `crca_xenium.h5ad` (27GB, 3.7M cells, 15 patients, 37 tissue sections).

Target size: matches our existing datasets (Pancreas 16,382 / Immune 30k / Lung 30k / NSCLC spatial 28,804 cells).

Already confirmed: `Niche` (6 categories, NicheCompass-derived) and `CN` (11 categories, finer split of the same) are both present but neither matches our composition-clustering method — kept in obs for reference only, not used as the final niche label. `celltype` (13 categories) is used as-is.

**Every cell below is safe to re-run**: each checks whether its output already exists (locally or on Drive) and skips the expensive work if so, instead of redoing a download/computation that already succeeded.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/data/spatial'
CRC_DIR = os.path.join(DATA_DIR, 'crc_xenium')
os.makedirs(CRC_DIR, exist_ok=True)
out_path = f'{CRC_DIR}/crc_xenium_prepped.h5ad'
print('CRC data dir:', CRC_DIR)
print('Final output path:', out_path, '-- exists:', os.path.exists(out_path))

# Pin numpy/pandas to Colab's preinstalled-compatible ranges -- plain
# "pip install anndata" pulls numpy 2.5+ which breaks scipy/numba already
# loaded in this runtime. If you still hit an AttributeError on
# numpy._core after this, do Runtime > Restart session once, then re-run
# this cell (the broken numpy is cached in the current process).
!pip install -q fsspec aiohttp h5py anndata "numpy<2.1,>=1.22" "pandas==2.2.2"

import fsspec, h5py, pandas as pd, numpy as np, scipy.sparse as sp, anndata as ad, requests

CRC_URL = 'https://ftp.ebi.ac.uk/biostudies/fire/S-BIAD/208/S-BIAD2208/Files/xenium/processed/crca_xenium.h5ad'
CRC_URL_EXPECTED_BYTES = 27337886004  # confirmed via HEAD request earlier

if os.path.exists(out_path):
    print('\nFinal prepared dataset already exists -- skipping remote h5ad open entirely.')
    h5 = None
else:
    of = fsspec.open(CRC_URL, mode='rb', block_size=4*1024*1024)
    f = of.open()
    h5 = h5py.File(f, 'r')
    print('opened remote h5ad. top-level:', list(h5.keys()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CRC data dir: /content/drive/MyDrive/data/spatial/crc_xenium
Final output path: /content/drive/MyDrive/data/spatial/crc_xenium/crc_xenium_prepped.h5ad -- exists: False
opened remote h5ad. top-level: ['X', 'layers', 'obs', 'obsm', 'obsp', 'uns', 'var', 'varm', 'varp']


## Step 1: read obs metadata for all cells (small — no expression data touched)

Skipped automatically if the final dataset already exists (nothing downstream needs `obs_df` in that case).

In [5]:
def read_obs_col(h5, col):
    node = h5['obs'][col]
    if isinstance(node, h5py.Group):
        cats = node['categories'][:]
        cats = [c.decode() if isinstance(c, bytes) else c for c in cats]
        codes = node['codes'][:]
        return pd.Categorical.from_codes(codes, categories=cats)
    else:
        return node[:]

if os.path.exists(out_path):
    print('Final prepared dataset already exists -- skipping obs metadata read.')
else:
    print('Reading obs metadata for all cells (small columns only, no expression matrix touched)...')
    obs_cols = ['patient_id', 'region', 'slide', 'tissue_region', 'celltype', 'Niche', 'CN']
    obs_df = pd.DataFrame({c: read_obs_col(h5, c) for c in obs_cols})
    print(f'Total cells in file: {len(obs_df)}')
    print()
    print('--- cells per patient_id ---')
    print(obs_df['patient_id'].value_counts())
    print()
    print('--- patient_id x tissue_region crosstab ---')
    print(pd.crosstab(obs_df['patient_id'], obs_df['tissue_region']))

Reading obs metadata for all cells (small columns only, no expression matrix touched)...
Total cells in file: 3706544

--- cells per patient_id ---
patient_id
m    440007
a    399055
n    324289
b    307719
j    293805
f    252389
d    246334
e    242757
k    239098
h    196921
l    194492
i    180891
c    170203
o    150730
g     67854
Name: count, dtype: int64

--- patient_id x tissue_region crosstab ---
tissue_region    core  margin  normal
patient_id                           
a              215437  183618       0
b              134610  173109       0
c               85936   84267       0
d              109874  103672   32788
e               61051   90655   91051
f              126097   88757   37535
g                7012       0   60842
h               88845       0  108076
i              131843       0   49048
j              129067  105595   59143
k               86146  110402   42550
l              124331   59831   10330
m              198454  136793  104760
n              16698

## Step 2: read X, var, obsm['spatial'], layers['counts'] remotely (no full-file download, no full-object load)

`ad.read_h5ad()` was crashing on RAM because it loads the *entire* file — including `layers` (multiple full-size matrix copies) and `obsp` (pairwise/neighbor graphs for 3.7M cells) and 6 other unused `obsm` embeddings — none of which we need. Reading only `X`/`var`/`obsm['spatial']`/`layers['counts']` directly via `h5py` keeps this to a few GB instead.

**Important:** `X` in this file is already log-normalized (`uns['log1p']` present, values are non-integer floats like 0.53, 2.17...), not raw counts. We separately pull `layers['counts']` (raw counts) too — needed later for the DGE/pseudobulk ground-truth step, which sums raw counts per metacell, not log-normalized values. Same convention as `s28nsc`: `X`=log-normalized (+ `"normalized": True` in its dataset config) and `layers['counts']`=raw for DGE.

In [ ]:
import time
# Read only what we need directly (X, var, obsm['spatial'], layers['counts'])
# via h5py -- skips obsp/unused obsm/other layers entirely, so this stays
# well under a few GB instead of the ad.read_h5ad() full-object load that
# blew RAM. X here is already log-normalized (uns['log1p'] present, values
# are non-integer) -- we ALSO need layers['counts'] (raw counts) separately,
# since build_pseudobulk (DGE ground-truth step) needs raw counts to sum,
# not log-normalized values. Matches s28nsc's own convention: X=log-normalized
# + "normalized": True in its dataset config + layers['counts']=raw for DGE.
if os.path.exists(out_path):
    print('Final prepared dataset already exists -- skipping remote X/var/spatial/counts read.')
    full = None
else:
    def read_full_csr(grp, label):
        shape = tuple(int(v) for v in grp.attrs['shape'])
        print(f'Reading {label} (data/indices/indptr) in full, shape={shape} ...')
        t0 = time.time()
        indptr = grp['indptr'][:]
        data = grp['data'][:]
        indices = grp['indices'][:]
        print(f'  read {label} in {time.time()-t0:.1f}s  nnz={len(data)}  '
              f'data+indices size={(data.nbytes+indices.nbytes)/1e9:.2f} GB')
        return sp.csr_matrix((data, indices, indptr), shape=shape)

    X_full = read_full_csr(h5['X'], 'X (log-normalized)')
    counts_full = read_full_csr(h5['layers']['counts'], 'layers/counts (raw)')

    var_names = h5['var']['_index'][:]
    var_names = [v.decode() if isinstance(v, bytes) else v for v in var_names]
    print(f'n_genes = {len(var_names)} (panel size sanity check, expect ~380)')

    print('Reading obsm[spatial] (small, 3.7M x 2)...')
    spatial_full = h5['obsm']['spatial'][:]

    full = ad.AnnData(X=X_full, obs=obs_df, var=pd.DataFrame(index=var_names))
    full.obsm['spatial'] = spatial_full
    full.layers['counts'] = counts_full
    del X_full, counts_full, spatial_full
    print('\nBuilt minimal AnnData (X + obs + var + spatial + layers[counts] only):', full)
    print('X memory:', f'{(full.X.data.nbytes + full.X.indices.nbytes + full.X.indptr.nbytes)/1e9:.2f} GB')
    print('counts memory:', f'{(full.layers["counts"].data.nbytes + full.layers["counts"].indices.nbytes + full.layers["counts"].indptr.nbytes)/1e9:.2f} GB')
    print('X sample values (should be non-integer, log-normalized):', full.X.data[:10])
    print('counts sample values (should be integers, raw):', full.layers['counts'].data[:10])
    print('\ncelltype counts:\n', full.obs['celltype'].value_counts())
    print('\npatient_id counts:\n', full.obs['patient_id'].value_counts())
    print('\ntissue_region counts:\n', full.obs['tissue_region'].value_counts())

## Step 3: select whole tissue sections (preserves spatial neighborhoods)

Random per-cell subsampling (the old version of this step) scatters cells across patients, destroying local spatial neighborhoods — breaks BANKSY and our own spatial-neighbor affinity graph, both of which need real neighbors present. Instead: pick whole sections (patient_id × tissue_region), keep every cell in each, greedily preferring sections that add a new patient or new tissue_region, stopping once total ≥ 30,000.

Given this dataset's actual section sizes (gap between ~10k and ~33k), that lands on 3 sections / ~50k cells / 3 of 15 patients / 2 of 3 tissue regions (core, normal — no margin, since the smallest margin section alone is 59k).

**Note:** the dataset saved on Drive from the old (random-subsample) version is now stale. Delete it once before re-running Steps 1-4 with this new logic:
```python
import os
if os.path.exists(out_path):
    os.remove(out_path)
```

In [ ]:
# Select whole tissue sections (patient_id x tissue_region), keeping every
# cell in each chosen section -- preserves real spatial neighborhoods
# (needed for BANKSY and for scProto's own spatial affinity graph; random
# cross-patient subsampling would scatter cells and destroy local
# neighborhoods). Greedily adds sections in ascending size order, preferring
# ones that bring in a new patient or new tissue_region, until total cells
# reaches TARGET_N_CELLS -- given this dataset's size distribution (a gap
# between ~10k and ~33k per section) that lands us around 50k, not exactly 30k.
TARGET_N_CELLS = 30000

if os.path.exists(out_path):
    print('Final prepared dataset already exists -- loading it instead of resampling.')
    adata = ad.read_h5ad(out_path)
    print(adata)
else:
    section_info = []
    for (patient, tregion), sub in full.obs.groupby(['patient_id', 'tissue_region'], observed=True):
        section_info.append({
            'section': f'{patient}_{tregion}',
            'patient_id': patient,
            'tissue_region': tregion,
            'n_cells': len(sub),
        })
    section_df = pd.DataFrame(section_info).sort_values('n_cells').reset_index(drop=True)
    print('--- all sections, smallest first ---')
    print(section_df.to_string(index=False))

    chosen = []
    seen_patients = set()
    seen_tissue_regions = set()
    total = 0
    for _, row in section_df.iterrows():
        if total >= TARGET_N_CELLS:
            break
        adds_diversity = (row['patient_id'] not in seen_patients) or (row['tissue_region'] not in seen_tissue_regions)
        if adds_diversity:
            chosen.append(row)
            seen_patients.add(row['patient_id'])
            seen_tissue_regions.add(row['tissue_region'])
            total += row['n_cells']

    print(f'\n--- CHOSEN sections ({len(chosen)}), total cells = {total} ---')
    for row in chosen:
        print(f"  {row['section']:12s} patient={row['patient_id']:3s} tissue_region={row['tissue_region']:8s} n_cells={row['n_cells']:6d}")
    print(f'\nPatients covered: {sorted(seen_patients)}')
    print(f'Tissue regions covered: {sorted(seen_tissue_regions)}')

    chosen_labels = {c['section'] for c in chosen}
    section_labels = full.obs['patient_id'].astype(str) + '_' + full.obs['tissue_region'].astype(str)
    keep_mask = section_labels.isin(chosen_labels).to_numpy()
    adata = full[keep_mask].copy()

    print('\n=== FINAL prepared dataset (whole sections, spatial neighborhoods intact) ===')
    print(adata)
    print('\ncelltype counts:\n', adata.obs['celltype'].value_counts())
    print('\nNiche counts:\n', adata.obs['Niche'].value_counts())
    print('\npatient_id (batch) counts:\n', adata.obs['patient_id'].value_counts())
    print('\ntissue_region counts:\n', adata.obs['tissue_region'].value_counts())
    assert adata.n_obs == total, f'expected {total} cells, got {adata.n_obs}'
    assert not adata.obs['celltype'].isna().any()
    assert not adata.obs['Niche'].isna().any()
    print('\nsanity checks passed: cell count matches chosen sections exactly, no missing celltype/Niche labels.')

    del full
    print('\nfreed the full in-memory dataset')

## Step 4: save the prepared (small) dataset to Drive

Skips the write if already saved, but always reloads + prints it so you get confirmation either way.

In [8]:
if os.path.exists(out_path):
    print(f'{out_path} already exists -- not overwriting.')
else:
    adata.write_h5ad(out_path)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f'Saved to {out_path} ({size_mb:.1f} MB)')

check = ad.read_h5ad(out_path)
print('\n--- current saved dataset ---')
print(check)
assert check.n_obs == adata.n_obs and check.n_vars == adata.n_vars
assert list(check.obs.columns) == list(adata.obs.columns)
print('OK: shape and columns match.')

Saved to /content/drive/MyDrive/data/spatial/crc_xenium/crc_xenium_prepped.h5ad (16.7 MB)

--- current saved dataset ---
AnnData object with n_obs × n_vars = 30907 × 380
    obs: 'patient_id', 'region', 'slide', 'tissue_region', 'celltype', 'Niche', 'CN'
    obsm: 'spatial'
OK: shape and columns match.


## Report back

After running, paste me:
1. Step 2's download/load confirmation (or "already exists, skipped")
2. The final celltype / Niche / patient_id / tissue_region counts from Step 3
3. The final saved file size / confirmation from Step 4